# 05 · Riesgo, relaciones entre activos y simulación financiera

**Financial Analytics Lab · Python · pandas · NumPy · SciPy · matplotlib**

**Pregunta orientadora:** ¿cómo se relacionan los rendimientos de distintos activos y cómo podemos examinar escenarios de riesgo de un portafolio?

En los módulos 01 a 04 aprendimos a representar inversiones, preparar datos, describir rendimientos y estudiar su evolución temporal. En este módulo examinaremos la relación entre activos y la incertidumbre de los resultados.

**Objetivos:** interpretar covarianza y correlación; calcular rendimiento y volatilidad de un portafolio con pesos; comprender una regresión lineal sencilla; estudiar cuantiles y una distribución normal como aproximación; ejecutar una simulación de Monte Carlo reproducible; y explicar sus supuestos y limitaciones.

**Trabajo autónomo:** ejecuta las celdas en orden, completa las prácticas y el reto integrador. No se requiere conocimiento previo de modelos avanzados, pero sí las bases de los módulos anteriores. Los resultados históricos y simulados no son predicciones ni recomendaciones de inversión.

## 0 · Problema de análisis

Un analista observa que tres activos tienen rendimientos diferentes. Sin embargo, para comprender el riesgo de una inversión conjunta necesita saber **cómo varían entre sí**. Si dos activos tienden a subir y bajar al mismo tiempo, su combinación puede comportarse de manera distinta a la de dos activos con movimientos menos sincronizados.

Analizaremos MSFT, AAPL y JPM durante un período histórico explícito. Para fines didácticos construiremos un portafolio con pesos de 40 %, 35 % y 25 %.

**Preguntas:** ¿qué activos se mueven de forma semejante?, ¿cómo se calcula la variabilidad del portafolio?, ¿qué muestra una regresión entre rendimientos?, ¿qué escenarios surgen bajo supuestos probabilísticos?

**Precaución:** correlación no implica causalidad; la diversificación no elimina todas las pérdidas; y las simulaciones dependen de supuestos que pueden no cumplirse.

## 1 · Recuperar y validar rendimientos históricos

Reutilizamos el enfoque ETL del módulo 02. Puedes cargar `precios_portafolio.csv` o consultar precios con `yfinance`. La fuente externa puede cambiar; por eso registramos fechas y parámetros.

La covarianza y la correlación requieren observaciones **alineadas por fecha**. No deben compararse rendimientos de días distintos como si fueran simultáneos.

In [ ]:
%pip -q install yfinance pandas numpy scipy matplotlib

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import yfinance as yf
from scipy import stats
from datetime import datetime, timezone

TICKERS = ["MSFT", "AAPL", "JPM"]
INICIO = "2023-01-01"
FIN = "2025-01-01"
USAR_CSV = False
print("Consulta UTC:", datetime.now(timezone.utc).isoformat(timespec="seconds"))

In [ ]:
if USAR_CSV:
    from google.colab import files
    archivos = files.upload()
    if "precios_portafolio.csv" not in archivos:
        raise FileNotFoundError("Sube precios_portafolio.csv.")
    precios = pd.read_csv("precios_portafolio.csv", index_col="Fecha", parse_dates=["Fecha"])
else:
    partes = []
    for ticker in TICKERS:
        tabla = yf.download(ticker, start=INICIO, end=FIN,
                            auto_adjust=True, progress=False, multi_level_index=False)
        if tabla.empty or "Close" not in tabla.columns:
            raise RuntimeError(f"No fue posible consultar {ticker}.")
        partes.append(tabla[["Close"]].rename(columns={"Close": ticker}))
    precios = pd.concat(partes, axis=1, join="inner")

precios.index = pd.to_datetime(precios.index)
precios = precios.sort_index().loc[lambda x: ~x.index.duplicated(keep="first")]
precios = precios.apply(pd.to_numeric, errors="coerce").dropna(how="any")
precios = precios.loc[(precios > 0).all(axis=1)]
rendimientos = precios.pct_change(fill_method=None).dropna(how="any")
if len(rendimientos) < 60 or rendimientos.shape[1] != 3:
    raise ValueError("Se requieren tres activos y al menos 60 rendimientos alineados.")
print("Observaciones de rendimientos:", len(rendimientos))
display((rendimientos.head() * 100).round(3))

**Tu práctica 1:** comprueba que las tres columnas correspondan a activos diferentes, que las fechas estén ordenadas y que no existan valores faltantes. Explica por qué utilizamos rendimientos en lugar de comparar los precios nominales.

In [ ]:
# TU PRÁCTICA 1
# Comprueba fechas, columnas, duplicados y datos faltantes.

## 2 · Covarianza: variaciones conjuntas

La **covarianza muestral** resume cómo se desvían conjuntamente dos series respecto de sus medias:

\[\operatorname{Cov}(X,Y)=\frac{\sum_{i=1}^{n}(x_i-\bar x)(y_i-\bar y)}{n-1}\]

Una covarianza positiva indica que las desviaciones tienden a compartir signo; una negativa, que tienden a tener signos opuestos. Su magnitud depende de las unidades y escalas, por lo que es difícil comparar pares únicamente mediante covarianzas.

In [ ]:
covarianzas = rendimientos.cov()
display(covarianzas.round(8))
x = rendimientos.iloc[:5, 0].to_numpy()
y = rendimientos.iloc[:5, 1].to_numpy()
cov_manual = ((x - x.mean()) * (y - y.mean())).sum() / (len(x) - 1)
cov_numpy = np.cov(x, y, ddof=1)[0, 1]
assert np.isclose(cov_manual, cov_numpy)
print("Covarianza de cinco observaciones comprobada:", cov_manual)

**Tu práctica 2:** selecciona dos activos y explica el signo de su covarianza. Comprueba manualmente la covarianza de cinco observaciones diferentes.

In [ ]:
# TU PRÁCTICA 2
# Verifica una covarianza y escribe una interpretación.

## 3 · Correlación: intensidad de la relación lineal

La **correlación de Pearson** estandariza la covarianza al dividirla entre las desviaciones estándar:

\[\rho_{XY}=\frac{\operatorname{Cov}(X,Y)}{s_Xs_Y}\]

Sus valores se sitúan entre −1 y 1. Valores cercanos a 1 indican relación lineal positiva fuerte; cercanos a −1, negativa fuerte; y cercanos a cero, relación lineal débil. **Correlación cero no implica independencia** y correlación no demuestra causalidad.

In [ ]:
correlaciones = rendimientos.corr()
display(correlaciones.round(3))
fig, ax = plt.subplots(figsize=(6, 5))
imagen = ax.imshow(correlaciones.to_numpy(), vmin=-1, vmax=1, cmap="coolwarm")
ax.set_xticks(range(len(correlaciones.columns)), correlaciones.columns)
ax.set_yticks(range(len(correlaciones.index)), correlaciones.index)
for i in range(len(correlaciones)):
    for j in range(len(correlaciones)):
        ax.text(j, i, f"{correlaciones.iloc[i, j]:.2f}", ha="center", va="center")
ax.set_title("Correlación de rendimientos")
fig.colorbar(imagen, ax=ax, label="Coeficiente de Pearson")
plt.tight_layout()
plt.show()

**Tu práctica 3:** identifica el par con mayor correlación fuera de la diagonal y el par con menor correlación. Explica por qué una relación menos positiva podría contribuir a la diversificación, sin garantizarla.

In [ ]:
# TU PRÁCTICA 3
# Compara los pares de activos y redacta una conclusión.

## 4 · Rendimiento de un portafolio ponderado

Un portafolio asigna una **ponderación** a cada activo. Si los pesos suman 1, el rendimiento por observación es la suma ponderada de los rendimientos individuales:

\[r_p=\sum_{i=1}^{n}w_i r_i\]

Este cálculo representa un portafolio con **pesos constantes reequilibrados en cada observación**. Si se compran cantidades fijas y no se reequilibra, los pesos cambian con los precios y el cálculo requiere otro procedimiento.

In [ ]:
pesos = pd.Series({"MSFT": 0.40, "AAPL": 0.35, "JPM": 0.25})
if not set(pesos.index).issubset(rendimientos.columns):
    raise ValueError("El CSV debe contener MSFT, AAPL y JPM para usar estos pesos.")
pesos = pesos.reindex(rendimientos.columns)
assert np.isclose(pesos.sum(), 1.0)
rend_portafolio = rendimientos.dot(pesos)
media_portafolio = rend_portafolio.mean()
print("Pesos:", pesos.to_dict())
print(f"Media por observación: {media_portafolio * 100:.4f}%")
manual_primera = sum(rendimientos.iloc[0][t] * pesos[t] for t in pesos.index)
assert np.isclose(manual_primera, rend_portafolio.iloc[0])
print("Primera observación contrastada.")

**Tu práctica 4:** prueba pesos de 50 %, 30 % y 20 %. Calcula manualmente el rendimiento del portafolio en la primera observación. Explica por qué los pesos deben sumar 100 %.

In [ ]:
# TU PRÁCTICA 4
# Define pesos alternativos y contrasta el primer rendimiento.

## 5 · Varianza y volatilidad del portafolio

La varianza del portafolio depende tanto de la variabilidad individual como de las **covarianzas**:

\[\sigma_p^2=\mathbf{w}^{T}\mathbf{\Sigma}\mathbf{w}\]

Aquí **w** es el vector de pesos y **Σ** la matriz de covarianzas. La volatilidad por observación es la raíz cuadrada de esa varianza.

Este cálculo muestra por qué sumar desviaciones estándar ponderadas no es una forma general correcta de calcular la volatilidad de un portafolio.

In [ ]:
matriz_cov = rendimientos.cov()
varianza_portafolio = float(pesos.to_numpy() @ matriz_cov.to_numpy() @ pesos.to_numpy())
volatilidad_portafolio = np.sqrt(varianza_portafolio)
volatilidad_directa = rend_portafolio.std(ddof=1)
assert np.isclose(volatilidad_portafolio, volatilidad_directa)
print(f"Volatilidad del portafolio: {volatilidad_portafolio * 100:.4f}% por observación")
print(f"Comprobación directa: {volatilidad_directa * 100:.4f}%")

**Tu práctica 5:** compara la volatilidad de tu portafolio alternativo con la del ejemplo. ¿Qué cambió? Evita afirmar que una ponderación es óptima sin establecer objetivos y restricciones.

In [ ]:
# TU PRÁCTICA 5
# Recalcula la volatilidad con los pesos de tu práctica anterior.

## 6 · Regresión lineal simple: estudiar una relación

Una **regresión lineal simple** aproxima la relación entre una variable dependiente Y y otra explicativa X mediante:

\[Y=\alpha+\beta X+\varepsilon\]

La pendiente **β** expresa el cambio promedio estimado en Y asociado con una unidad adicional de X dentro del modelo. El intercepto **α** es el valor estimado de Y cuando X es cero. El coeficiente **R²** resume la proporción de variación de Y explicada por la relación lineal ajustada.

Analizaremos los rendimientos de MSFT (Y) frente a AAPL (X). **No interpretaremos la regresión como una relación causal**.

In [ ]:
x = rendimientos["AAPL"].to_numpy()
y = rendimientos["MSFT"].to_numpy()
ajuste = stats.linregress(x, y)
print(f"Intercepto: {ajuste.intercept:.6f}")
print(f"Pendiente beta: {ajuste.slope:.4f}")
print(f"R²: {ajuste.rvalue**2:.4f}")
fig, ax = plt.subplots(figsize=(8, 5))
ax.scatter(x * 100, y * 100, alpha=0.35, s=14)
orden = np.argsort(x)
ax.plot(x[orden] * 100, (ajuste.intercept + ajuste.slope * x[orden]) * 100)
ax.set(title="Relación lineal entre rendimientos", xlabel="AAPL (%)", ylabel="MSFT (%)")
ax.grid(alpha=0.25)
plt.tight_layout()
plt.show()

**Tu práctica 6:** explica con tus palabras el significado de la pendiente y de R². ¿Qué no podemos concluir sobre causalidad o sobre el comportamiento futuro de los activos?

In [ ]:
# TU PRÁCTICA 6
# Interpreta beta y R² con las cifras obtenidas.

## 7 · Distribuciones de probabilidad y cuantiles

Una distribución de probabilidad representa posibles resultados y sus probabilidades bajo un modelo. La **distribución normal** se caracteriza por media y desviación estándar, pero los rendimientos financieros reales pueden presentar asimetría y colas más pronunciadas.

Un **cuantil empírico** se calcula a partir de los datos observados; un **cuantil paramétrico** proviene de una distribución supuesta. Compararlos ayuda a reconocer que el modelo probabilístico no es idéntico a la muestra.

In [ ]:
media = rend_portafolio.mean()
desv = rend_portafolio.std(ddof=1)
percentil_empirico_5 = rend_portafolio.quantile(0.05)
percentil_normal_5 = stats.norm.ppf(0.05, loc=media, scale=desv)
print(f"Percentil empírico 5 %: {percentil_empirico_5 * 100:.3f}%")
print(f"Percentil normal 5 %: {percentil_normal_5 * 100:.3f}%")
fig, ax = plt.subplots(figsize=(9, 4))
ax.hist(rend_portafolio * 100, bins=35, density=True, alpha=0.55, label="Rendimientos observados")
xx = np.linspace(rend_portafolio.min(), rend_portafolio.max(), 300)
ax.plot(xx * 100, stats.norm.pdf(xx, loc=media, scale=desv) / 100, label="Normal ajustada")
ax.set(title="Rendimientos observados y aproximación normal", xlabel="Rendimiento (%)", ylabel="Densidad")
ax.legend()
plt.tight_layout()
plt.show()

**Interpretación:** el percentil 5 % describe un umbral bajo de rendimientos; no significa que el 5 % de las inversiones vaya necesariamente a perder esa cantidad. La comparación entre cuantiles depende del período, la muestra y los supuestos.

**Tu práctica 7:** compara los dos percentiles y explica por qué pueden diferir. ¿Qué limitación introduce suponer normalidad?

In [ ]:
# TU PRÁCTICA 7
# Compara cuantiles y comenta las limitaciones.

## 8 · Monte Carlo: explorar escenarios bajo supuestos

Una **simulación de Monte Carlo** utiliza números aleatorios para construir escenarios posibles de acuerdo con un modelo. No reproduce necesariamente el comportamiento real de los mercados.

Simularemos **20 sesiones** de rendimientos del portafolio bajo una distribución normal independiente con media y desviación estándar históricas constantes. Utilizaremos una semilla para obtener resultados reproducibles.

En cada trayectoria el valor inicial es 100 y evoluciona multiplicando factores (1 + rendimiento). El modelo no contempla cambios de régimen, costos, colas extremas ni dependencia temporal.

In [ ]:
SEMILLA = 2026
TRAYECTORIAS = 2000
HORIZONTE = 20
rng = np.random.default_rng(SEMILLA)
simulados = rng.normal(loc=media, scale=desv, size=(HORIZONTE, TRAYECTORIAS))
valores = np.vstack([np.full(TRAYECTORIAS, 100.0),
                     100.0 * np.cumprod(1 + simulados, axis=0)])
valores_finales = valores[-1]
print("Media de valores finales:", round(valores_finales.mean(), 2))
print("Percentil 5 de valores finales:", round(np.quantile(valores_finales, 0.05), 2))
print("Proporción simulada por debajo de 100:", round(np.mean(valores_finales < 100), 3))
fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(valores[:, :60], alpha=0.3, linewidth=0.8)
ax.axhline(100, linewidth=1)
ax.set(title="Escenarios simulados: 20 sesiones", xlabel="Sesiones", ylabel="Valor hipotético (inicio = 100)")
ax.grid(alpha=0.25)
plt.tight_layout()
plt.show()

**Interpretación:** los resultados describen escenarios **del modelo**, no probabilidades futuras verificadas. Una semilla fija permite repetir la simulación, pero no elimina la incertidumbre del modelo.

**Tu práctica 8:** cambia la semilla y compara resultados; después aumenta el horizonte a 40 sesiones. Explica por qué la dispersión de los valores finales puede cambiar.

In [ ]:
# TU PRÁCTICA 8
# Prueba otra semilla y horizonte; registra los cambios.

## 9 · Síntesis: indicadores complementarios

La correlación describe relaciones lineales entre activos; la volatilidad del portafolio combina pesos y covarianzas; la regresión caracteriza una asociación estadística; los cuantiles y las simulaciones examinan resultados bajo distintos enfoques.

Ninguno de estos procedimientos, por separado, determina qué inversión conviene realizar. La interpretación exige conocer el período, la calidad de los datos, los supuestos y los objetivos financieros.

In [ ]:
resumen = pd.DataFrame([{
    "Media portafolio (%)": media * 100,
    "Volatilidad por observación (%)": desv * 100,
    "Percentil empírico 5 (%)": percentil_empirico_5 * 100,
    "Percentil normal 5 (%)": percentil_normal_5 * 100,
    "R² MSFT~AAPL": ajuste.rvalue ** 2,
    "Percentil 5 valor simulado": np.quantile(valores_finales, 0.05)
}])
display(resumen.round(4))
resumen.to_csv("resumen_riesgo_simulacion.csv", index=False)
print("Archivo exportado: resumen_riesgo_simulacion.csv")

## 10 · Reto integrador: análisis de riesgo de un portafolio

Construye un estudio reproducible de tres activos. Debe incluir:

1. Selección y justificación de activos, fuente y período.
2. Preparación de rendimientos alineados por fecha.
3. Matriz de covarianza y correlación, con una comprobación manual.
4. Portafolio ponderado con pesos que sumen 100 %.
5. Rendimiento y volatilidad del portafolio, con verificación de la fórmula matricial.
6. Regresión lineal simple entre dos activos e interpretación de β y R².
7. Percentiles empírico y normal, con discusión del supuesto de normalidad.
8. Simulación reproducible de Monte Carlo con al menos 1.000 trayectorias.
9. Tres visualizaciones interpretadas y una tabla resumen exportada.
10. Conclusión de **250 a 350 palabras** con resultados, supuestos, limitaciones y fuentes.

**Criterio de calidad:** no presentes una simulación como predicción ni una correlación como causalidad. Explica el uso de herramientas de IA y recursos externos.

In [ ]:
# RETO INTEGRADOR — TU SOLUCIÓN ORIGINAL
# Construye tus indicadores, verificaciones, gráficos y conclusiones.

## 11 · Autoevaluación y continuidad

- [ ] Distingo covarianza de correlación.
- [ ] Interpreto relaciones entre activos sin atribuir causalidad.
- [ ] Calculo rendimientos ponderados y volatilidad de un portafolio.
- [ ] Comprendo la pendiente y R² de una regresión simple.
- [ ] Distingo cuantiles empíricos de cuantiles bajo un modelo.
- [ ] Ejecuto una simulación con semilla reproducible.
- [ ] Reconozco los límites de las distribuciones y simulaciones.

**Próximo módulo:** integraremos los indicadores aprendidos en una aplicación financiera interactiva desarrollada progresivamente con Streamlit.

**Documentación:** [pandas](https://pandas.pydata.org/docs/) · [NumPy](https://numpy.org/doc/stable/) · [SciPy](https://docs.scipy.org/doc/scipy/) · [yfinance](https://ranaroussi.github.io/yfinance/) · [Streamlit](https://docs.streamlit.io/).

Los resultados son académicos y no constituyen recomendaciones de inversión.